# 🎚️ Notebook 2 — Gradual Traffic Shift, Dark Launches & Rollback

In Notebook 1 the facade routed 100% or 0% of a path. In the real world you
**ramp up slowly**: 1% → 5% → 25% → 50% → 100%, watching metrics between each step.

This notebook covers three powerful techniques:

1. **Canary routing** — send a small % of real traffic to the new service.
2. **Dark launch / shadow traffic** — send the *same* request to both, compare answers, but only return the legacy answer to the user.
3. **Kill switch / instant rollback** — one config flip sends 100% back to legacy.


## 🛠️ Setup

```bash
cd 05-microservices/strangler
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> This lab uses **only the Python standard library** — no servers, no databases. Every cell is a small simulation you can read end-to-end.


## 1. ❌ Bad practice: flip the switch for everyone at once

```python
# On Monday morning:
MIGRATED_PREFIXES = ("/users",)   # 100% of /users traffic -> new service
```

Problems:
- Any bug in the new service hits **every** user immediately.
- You have no gradient of "5 customers complained" vs "5000 customers complained".
- The only rollback is a full redeploy.

We can do better by making the router choose *probabilistically*.

## 2. ✅ Canary routing: ramp up by percentage

In [ ]:
import random, collections

def legacy(path):     return {"source": "legacy", "path": path, "v": 1}
def new_service(path): return {"source": "new",    "path": path, "v": 2}

class Router:
    """Percent-based router. Think of it as a feature-flag per path prefix."""
    def __init__(self):
        self.percent_new = {}   # prefix -> 0..100

    def set_percent(self, prefix, pct):
        assert 0 <= pct <= 100
        self.percent_new[prefix] = pct

    def handle(self, path):
        for prefix, pct in self.percent_new.items():
            if path.startswith(prefix):
                # random.randint(1,100) <= pct gives us ~pct% true
                return new_service(path) if random.randint(1, 100) <= pct else legacy(path)
        return legacy(path)

random.seed(0)
r = Router()

for pct in (1, 10, 50, 100):
    r.set_percent("/users", pct)
    counts = collections.Counter(r.handle("/users/42")["source"] for _ in range(2000))
    print(f"{pct:3d}% canary -> {dict(counts)}")


### Reading the output
At 1% only ~20 of 2000 requests go to the new service — just enough to catch
catastrophic bugs without risking everyone. At 100% the legacy service sees zero traffic
and can be retired.

A realistic ramp schedule might be:

| Day | % on new |
|----:|---------:|
| 1   | 1 %      |
| 2   | 5 %      |
| 3   | 25 %     |
| 4   | 50 %     |
| 7   | 100 %    |

…with automated rollback if error rate or latency exceeds a threshold.

## 3. 🕯️ Dark launch (a.k.a. shadow traffic)

> *"Run the new code in production with 100% of real traffic, but throw its
> output away — just compare it to legacy."*

This is the **safest** way to test a new service with real data. If the answers
differ, you log it and investigate. The user never sees the new system's answer
until you're confident.

In [ ]:
import random, time

# Pretend legacy and new agree on easy cases but new has a bug on id=13.
def legacy_user(user_id):
    return {"id": user_id, "name": f"user_{user_id}", "tier": "gold" if user_id < 100 else "silver"}

def new_user(user_id):
    # BUG: new service forgets the "silver" rule for id=13 specifically
    if user_id == 13:
        return {"id": user_id, "name": f"user_{user_id}", "tier": "gold"}
    return legacy_user(user_id)

def dark_launch(user_id, log):
    legacy_resp = legacy_user(user_id)
    shadow_resp = new_user(user_id)          # call new service too
    if legacy_resp != shadow_resp:
        log.append((user_id, legacy_resp, shadow_resp))
    return legacy_resp                        # user only ever sees legacy

mismatches = []
for uid in range(1, 201):
    dark_launch(uid, mismatches)

print(f"Requests compared: 200")
print(f"Mismatches found:  {len(mismatches)}")
for row in mismatches[:5]:
    print("  ", row)


We caught the bug on `id=13` **before** any real user saw a wrong answer.
This is how GitHub famously migrated their merge logic (project "Scientist",
open-sourced at [github/scientist](https://github.com/github/scientist)) and how
Stripe migrates API endpoints today.

Watch out for:
- **Side effects.** Don't dark-launch writes — two `POST /charge` calls mean
  you charged the customer twice. Dark-launch reads; for writes, use dual-write
  with idempotency keys (see Notebook 3).
- **Cost.** Every request now costs 2× CPU. Fine for a week of validation; not
  for forever.


## 4. 🚨 Kill switch: instant rollback

The whole point of the strangler pattern is that *any bad step is reversible in seconds*.

In [ ]:
class RouterWithKillSwitch(Router):
    def __init__(self):
        super().__init__()
        self.kill_switch = False    # when True: everything goes to legacy

    def handle(self, path):
        if self.kill_switch:
            return legacy(path)
        return super().handle(path)

r = RouterWithKillSwitch()
r.set_percent("/users", 100)   # we're "done" migrating

# Everything on new:
print("before kill:", r.handle("/users/1"))

# Incident! Flip the switch.
r.kill_switch = True
print("after kill :", r.handle("/users/1"))

# Fix the bug, flip it back.
r.kill_switch = False
print("recovered  :", r.handle("/users/1"))


In production the kill switch is a config value (LaunchDarkly, Unleash, a Redis
key, a column in a DB) that the router re-reads every few seconds. Engineers
on-call can flip it *without a deploy*.

### The unbreakable rule
**Never migrate a slice that you can't roll back from.** If you can't go back,
you're doing a big-bang wearing a costume.

## 5. 📊 Production checklist before ramping up

Before moving from 1% → 5%, compare these metrics between legacy and new:

- ✅ **Error rate** — 5xx responses, exceptions, timeouts. Must be **≤** legacy.
- ✅ **p50 / p95 / p99 latency** — tail latency often regresses first.
- ✅ **Result parity** — dark-launch diff rate below ~0.01%.
- ✅ **Business KPIs** — checkout conversion, sign-up rate, payment success. The
  most dangerous bugs don't show up as exceptions.
- ✅ **Cost** — CPU, memory, DB load on the new service at *projected* full traffic.

Ramp up only when *all* are green. Automate the rollback so the humans don't
have to argue at 3am.


### ➡️ Next
Notebook 3 puts it all together: a full banking migration with dual-write data
handling, per-endpoint progress, metrics, and the retirement phase.